# Milestone 3: Advanced Ship Detection Pipeline 🚢

This notebook implements the complete **Two-Stage Pipeline** for the Airbus Ship Detection Challenge.

### **Pipeline Steps:**
1.  **Data Optimization**: Pre-process Pandas DataFrames to avoid slow string parsing during training.
2.  **Stage 1 (Classifier)**: Train a lightweight ResNet34 to filter out empty images (Empty vs. Ship).
3.  **Stage 2 (Segmenter)**: Train a U-Net (EfficientNet-B4) specifically on images that contain ships.
4.  **Threshold Tuning**: Automatically find the optimal probability threshold to maximize the F2 Score.
5.  **Final Evaluation**: Test the full pipeline on unseen data (`test_split.csv`) and visualize results.

In [1]:
# 1. Setup & Imports
%matplotlib inline
%load_ext autoreload
%autoreload 2

import sys
import os
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm.notebook import tqdm
from ast import literal_eval
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Add the '../src' directory to python path so we can import our modules
sys.path.append(os.path.abspath('../src'))

# Custom Modules (from src folder)
from dataset import AirbusDataset
from model import ShipClassifier, ShipSegmenter
from losses import TverskyLoss, BCEDiceLoss
from transforms import get_transforms
from metrics import calculate_iou, AverageMeter
from rle import mask_to_rle

print(f"Torch Version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

Torch Version: 2.9.1+cu128
Device: cuda


In [2]:
# 2. Global Configuration
CONFIG = {
    "TRAIN_CSV": "../data/train_split.csv",
    "VAL_CSV": "../data/val_split.csv",
    "TEST_CSV": "../data/test_split.csv",
    "IMG_DIR": "../data/train_v2/",
    "DEVICE": "cuda" if torch.cuda.is_available() else "cpu", # <--- FIX: Remove outer quotes
    
    # Training Hyperparameters
    "IMG_SIZE": 768,
    "CLF_IMG_SIZE": 384,  # Smaller size for classifier speed
    "BATCH_SIZE": 8,      # Adjust based on VRAM (8 for 12GB, 16 for 24GB)
    "NUM_WORKERS": 4,
    "LR": 1e-4,
    
    # Stage settings
    "CLF_EPOCHS": 5,
    "SEG_EPOCHS": 15
}

# Helper class to track metrics
class MetricMonitor:
    def __init__(self):
        self.reset()
    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

## Stage 1: The Classifier (Ship vs. No Ship)
First, we train a simple binary classifier to filter out empty sea images. This drastically reduces false positives.

In [3]:
# --- 3.1 Prepare Balanced Data for Classifier ---
print("Loading Data for Classifier...")
train_df = pd.read_csv(CONFIG['TRAIN_CSV'])

# Balance the dataset (50% Ship, 50% Empty)
ships = train_df[train_df['HasShip'] == 1]
empty = train_df[train_df['HasShip'] == 0].sample(n=len(ships), random_state=42)
clf_df = pd.concat([ships, empty]).sample(frac=1).reset_index(drop=True)

print(f"Balanced Training Data: {len(clf_df)} images")

# Simple Dataset Wrapper for Classification
class ClassifierDataset(torch.utils.data.Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(os.path.join(self.img_dir, row['ImageId']))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        label = float(row['HasShip'])
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label).float().unsqueeze(0)

clf_transforms = A.Compose([
    A.Resize(CONFIG['CLF_IMG_SIZE'], CONFIG['CLF_IMG_SIZE']),
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5),
    A.Normalize(), ToTensorV2()
])

clf_loader = DataLoader(
    ClassifierDataset(clf_df, CONFIG['IMG_DIR'], transform=clf_transforms),
    batch_size=32, shuffle=True, num_workers=CONFIG['NUM_WORKERS']
)

Loading Data for Classifier...
Balanced Training Data: 12376 images


In [4]:
# --- 3.2 Train Classifier ---
print("Training Classifier (ResNet34)...")
classifier = ShipClassifier(backbone='resnet34').to(CONFIG['DEVICE'])
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-4)
scaler = GradScaler()

clf_history = []

for epoch in range(CONFIG['CLF_EPOCHS']):
    classifier.train()
    losses = MetricMonitor()
    accs = MetricMonitor()
    
    pbar = tqdm(clf_loader, desc=f"Clf Epoch {epoch+1}", leave=False)
    for imgs, labels in pbar:
        imgs, labels = imgs.to(CONFIG['DEVICE']), labels.to(CONFIG['DEVICE'])
        
        with autocast():
            out = classifier(imgs)
            loss = criterion(out, labels)
            
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Calc Acc
        preds = (torch.sigmoid(out) > 0.5).float()
        acc = (preds == labels).float().mean()
        
        losses.update(loss.item(), imgs.size(0))
        accs.update(acc.item(), imgs.size(0))
        pbar.set_postfix(loss=losses.avg, acc=accs.avg)
        
    print(f"Epoch {epoch+1}: Loss {losses.avg:.4f} | Acc {accs.avg:.4f}")
    clf_history.append(losses.avg)

torch.save(classifier.state_dict(), "../src/best_classifier.pth")
print("Classifier Saved!")

Training Classifier (ResNet34)...


/tmp/ipykernel_156879/1420448728.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Clf Epoch 1:   0%|          | 0/387 [00:00<?, ?it/s]

/tmp/ipykernel_156879/1420448728.py:19: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Loss 0.2149 | Acc 0.9186


...

Epoch 3: Loss 0.1183 | Acc 0.9598


Clf Epoch 4:   0%|          | 0/387 [00:00<?, ?it/s]

Epoch 4: Loss 0.1046 | Acc 0.9632


Clf Epoch 5:   0%|          | 0/387 [00:00<?, ?it/s]

Epoch 5: Loss 0.1004 | Acc 0.9662
Classifier Saved!


## Stage 2: The Segmenter (Pixel-Level)
Now we train the U-Net. **Optimization Note:** We convert the RLE columns from strings to lists *before* creating the Dataset. This prevents the data loader from becoming a CPU bottleneck.

In [5]:
# --- 4.1 Optimize Data Loading ---
def preprocess_df(df):
    # Convert string "['...']" to list object [...] ONCE here
    if len(df) > 0 and isinstance(df['RleMasks'].iloc[0], str):
        df['RleMasks'] = df['RleMasks'].apply(literal_eval)
    return df

print("Loading & Preprocessing Segmentation Data...")
train_full = pd.read_csv(CONFIG['TRAIN_CSV'])
val_full = pd.read_csv(CONFIG['VAL_CSV'])

# Filter: Only train segmenter on images WITH ships
train_seg_df = train_full[train_full['HasShip'] == 1].reset_index(drop=True)
val_seg_df = val_full[val_full['HasShip'] == 1].reset_index(drop=True)

# Apply Optimization
train_seg_df = preprocess_df(train_seg_df)
val_seg_df = preprocess_df(val_seg_df)

print(f"Train Size (Ships): {len(train_seg_df)} | Val Size (Ships): {len(val_seg_df)}")

# Create Datasets
train_ds = AirbusDataset(train_seg_df, CONFIG['IMG_DIR'], transform=get_transforms('train', CONFIG['IMG_SIZE']))
val_ds = AirbusDataset(val_seg_df, CONFIG['IMG_DIR'], transform=get_transforms('valid', CONFIG['IMG_SIZE']))

train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, 
                          num_workers=CONFIG['NUM_WORKERS'], pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG['BATCH_SIZE'], shuffle=False, 
                        num_workers=CONFIG['NUM_WORKERS'], pin_memory=True)

Loading & Preprocessing Segmentation Data...
Train Size (Ships): 6188 | Val Size (Ships): 1326


/home/punkostigyork/anaconda3/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [6]:
# --- 4.2 Helper Metrics Function ---
def calculate_metrics_batch(preds, targets, threshold=0.5):
    with torch.no_grad():
        preds = (torch.sigmoid(preds) > threshold).float()
        tp = (preds * targets).sum().item()
        fp = (preds * (1 - targets)).sum().item()
        fn = ((1 - preds) * targets).sum().item()
        
        eps = 1e-7
        prec = tp / (tp + fp + eps)
        rec = tp / (tp + fn + eps)
        f2 = (5 * prec * rec) / (4 * prec + rec + eps)
        return prec, rec, f2

In [7]:
# --- 4.3 Segmenter Training Loop ---
print("Initializing Segmenter (EfficientNet-B4 U-Net)...")
segmenter = ShipSegmenter(encoder_name='efficientnet-b4').to(CONFIG['DEVICE'])

# Using Tversky Loss (alpha=0.3, beta=0.7) to emphasize Recall (F2 Score)
seg_criterion = TverskyLoss(alpha=0.3, beta=0.7)
seg_optimizer = torch.optim.AdamW(segmenter.parameters(), lr=CONFIG['LR'])
seg_scaler = GradScaler()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(seg_optimizer, T_max=CONFIG['SEG_EPOCHS'])

history = {'train_loss': [], 'val_loss': [], 'val_f2': [], 'val_rec': []}
best_f2 = 0.0

print("Starting Segmentation Training...")

for epoch in range(CONFIG['SEG_EPOCHS']):
    segmenter.train()
    t_loss = MetricMonitor()
    
    # Train Step
    pbar = tqdm(train_loader, desc=f"Seg Epoch {epoch+1}", leave=False)
    for imgs, masks in pbar:
        imgs, masks = imgs.to(CONFIG['DEVICE']), masks.to(CONFIG['DEVICE'])
        
        with autocast():
            preds = segmenter(imgs)
            loss = seg_criterion(preds, masks)
            
        seg_optimizer.zero_grad()
        seg_scaler.scale(loss).backward()
        seg_scaler.step(seg_optimizer)
        seg_scaler.update()
        
        t_loss.update(loss.item(), imgs.size(0))
        pbar.set_postfix(loss=t_loss.avg)

    # Validation Step
    segmenter.eval()
    v_loss = MetricMonitor()
    v_f2 = MetricMonitor()
    v_rec = MetricMonitor()
    
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(CONFIG['DEVICE']), masks.to(CONFIG['DEVICE'])
            with autocast():
                preds = segmenter(imgs)
                loss = seg_criterion(preds, masks)
            
            _, rec, f2 = calculate_metrics_batch(preds, masks, threshold=0.5)
            v_loss.update(loss.item(), imgs.size(0))
            v_f2.update(f2, imgs.size(0))
            v_rec.update(rec, imgs.size(0))

    scheduler.step()
    
    # Logging
    history['train_loss'].append(t_loss.avg)
    history['val_loss'].append(v_loss.avg)
    history['val_f2'].append(v_f2.avg)
    history['val_rec'].append(v_rec.avg)
    
    print(f"Epoch {epoch+1} | Train Loss: {t_loss.avg:.4f} | Val Loss: {v_loss.avg:.4f} | Val F2: {v_f2.avg:.4f}")
    
    if v_f2.avg > best_f2:
        best_f2 = v_f2.avg
        torch.save(segmenter.state_dict(), "../src/best_ship_segmenter.pth")
        print("✅ Best Model Saved!")

Initializing Segmenter (EfficientNet-B4 U-Net)...


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Starting Segmentation Training...


/tmp/ipykernel_156879/3785870582.py:8: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  seg_scaler = GradScaler()


Seg Epoch 1:   0%|          | 0/774 [00:00<?, ?it/s]

/tmp/ipykernel_156879/3785870582.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 0 has a total capacity of 11.49 GiB of which 48.81 MiB is free. Including non-PyTorch memory, this process has 10.70 GiB memory in use. Of the allocated memory 10.35 GiB is allocated by PyTorch, and 134.05 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# --- 5. Visualize Learning Curves ---
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', marker='o')
plt.plot(history['val_loss'], label='Val Loss', marker='o')
plt.title('Loss Curves (Tversky)')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history['val_f2'], label='Val F2 Score', color='green', marker='o')
plt.plot(history['val_rec'], label='Val Recall', color='orange', linestyle='--')
plt.title('Metric Curves')
plt.legend()
plt.grid(True)
plt.show()

## Stage 3: Threshold Tuning
Since F2 Score prioritizes recall, the default threshold of `0.5` is rarely optimal. We sweep values from 0.1 to 0.9 to find the best cutoff.

In [ ]:
def find_best_threshold(model, loader):
    model.eval()
    print("Running threshold sweep on Validation set...")
    
    # Collect all predictions and targets on CPU to avoid VRAM overflow
    all_probs = []
    all_targets = []
    
    with torch.no_grad():
        for imgs, masks in tqdm(loader, leave=False):
            imgs = imgs.to(CONFIG['DEVICE'])
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu()
            all_probs.append(probs)
            all_targets.append(masks.cpu())
            
    all_probs = torch.cat(all_probs)
    all_targets = torch.cat(all_targets)
    
    # Sweep
    thresholds = np.arange(0.1, 0.9, 0.05)
    best_thresh = 0.5
    best_score = 0.0
    
    scores = []
    for t in thresholds:
        _, _, f2 = calculate_metrics_batch(all_probs, all_targets, threshold=t)
        scores.append(f2)
        if f2 > best_score:
            best_score = f2
            best_thresh = t
            
    plt.figure(figsize=(8, 4))
    plt.plot(thresholds, scores, marker='o')
    plt.title(f"Best Threshold: {best_thresh:.2f} (F2: {best_score:.4f})")
    plt.xlabel("Threshold")
    plt.ylabel("F2 Score")
    plt.grid(True)
    plt.show()
    
    return best_thresh

# Load best model and tune
segmenter.load_state_dict(torch.load("../src/best_ship_segmenter.pth"))
OPTIMAL_THRESHOLD = find_best_threshold(segmenter, val_loader)
print(f"Selected Optimal Threshold: {OPTIMAL_THRESHOLD}")

## Stage 4: Final Evaluation (Pipeline on Test Set)
Now we run the **Test Set** through the full pipeline:
1.  **Classifier:** Is it empty? If yes -> Mask is empty.
2.  **Segmenter:** If no -> Predict mask using optimal threshold.
3.  **Visualization:** Compare Pred vs. True.

In [ ]:
# --- 4.1 Load Test Data ---
test_df = pd.read_csv(CONFIG['TEST_CSV'])
test_df = preprocess_df(test_df) # Optimization

# Keep only ships for visualization fun, or all for accurate metric
# Let's check ships primarily to see performance
test_df_ships = test_df[test_df['HasShip'] == 1].reset_index(drop=True)
test_ds = AirbusDataset(test_df_ships, CONFIG['IMG_DIR'], transform=get_transforms('valid', CONFIG['IMG_SIZE']))
test_loader = DataLoader(test_ds, batch_size=8, shuffle=False, num_workers=4)

# --- 4.2 Pixel-wise Confusion Matrix ---
def plot_confusion_matrix_pixels(model, loader, threshold):
    model.eval()
    total_tp, total_fp, total_fn, total_tn = 0, 0, 0, 0
    
    print("Calculating Pixel Confusion Matrix on Test Set...")
    with torch.no_grad():
        for imgs, masks in tqdm(loader):
            imgs, masks = imgs.to(CONFIG['DEVICE']), masks.to(CONFIG['DEVICE'])
            preds = (torch.sigmoid(model(imgs)) > threshold).long()
            masks = masks.long()
            
            total_tp += (preds * masks).sum().item()
            total_fp += (preds * (1-masks)).sum().item()
            total_fn += ((1-preds) * masks).sum().item()
            # TN is usually huge (sea), can skip or count
            
    # Simple Matrix
    print(f"\nTP (Ship Correct): {total_tp}")
    print(f"FP (Ghost Ship):   {total_fp}")
    print(f"FN (Missed Ship):  {total_fn}")
    
    precision = total_tp / (total_tp + total_fp + 1e-6)
    recall = total_tp / (total_tp + total_fn + 1e-6)
    f2 = (5 * precision * recall) / (4 * precision + recall + 1e-6)
    
    print(f"\nTest Precision: {precision:.4f}")
    print(f"Test Recall:    {recall:.4f}")
    print(f"Test F2 Score:  {f2:.4f}")

plot_confusion_matrix_pixels(segmenter, test_loader, OPTIMAL_THRESHOLD)

In [ ]:
# --- 4.3 Visual Predictions ---
def show_predictions(model, loader, threshold, n=5):
    model.eval()
    imgs, masks = next(iter(loader))
    imgs = imgs.to(CONFIG['DEVICE'])
    
    with torch.no_grad():
        preds = torch.sigmoid(model(imgs))
        preds = (preds > threshold).float()
        
    imgs = imgs.cpu().permute(0, 2, 3, 1).numpy()
    masks = masks.cpu().squeeze().numpy()
    preds = preds.cpu().squeeze().numpy()
    
    # Denormalize images for display
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    
    plt.figure(figsize=(15, 5*n))
    for i in range(n):
        img = np.clip(imgs[i] * std + mean, 0, 1)
        
        plt.subplot(n, 3, i*3 + 1)
        plt.imshow(img)
        plt.title("Input Image")
        plt.axis('off')
        
        plt.subplot(n, 3, i*3 + 2)
        plt.imshow(masks[i], cmap='gray')
        plt.title("Ground Truth")
        plt.axis('off')
        
        plt.subplot(n, 3, i*3 + 3)
        plt.imshow(preds[i], cmap='gray')
        plt.title(f"Prediction (Thresh {threshold:.2f})")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visualizing Test Set Predictions...")
show_predictions(segmenter, test_loader, OPTIMAL_THRESHOLD, n=4)